In [ ]:
import psycopg2, os
import json
from postgres_utilities import connect_db, close_db
from s3_utilities import get_s3_connection_profile, S3ConnectionProfile

In [12]:
s3_connection_profile = get_s3_connection_profile()
#s3_connection_profile = S3ConnectionProfile("s3://minio-api-minio.apps.ai-dev01.kni.syseng.devcluster.openshift.com", 
#                                 "edb-aidb", "us-east-1", "false", "minio", "minio1234!", "recommender_images")

In [ ]:
def create_image_retriever(conn):

    with conn.cursor() as cur:
        # Drop existing objects if any
        cur.execute("DROP SERVER IF EXISTS images_s3_little CASCADE;")
        cur.execute("SELECT aidb.delete_model('recom_images');")
        cur.execute("SELECT aidb.delete_volume('images_bucket_vol');")

        try:
            cur.execute("SAVEPOINT risky_operation")
            cur.execute("SELECT aidb.delete_retriever('recom_images');")
        except Exception as e:
            cur.execute("ROLLBACK TO SAVEPOINT risky_operation") 

        cur.execute("SELECT pgfs.delete_storage_location('images_s3_little');")
        cur.execute("DROP TABLE IF EXISTS recom_images_vector CASCADE;")

        # Using EDB's pgfs extension, let's now connect to the in-cluster s3 bucket storage location
        options = json.dumps(
            {
                "endpoint": s3_connection_profile.endpoint_url,
                "region": s3_connection_profile.region,
            }
        )
        credentials = json.dumps(
            {
                "access_key_id": s3_connection_profile.access_key,
                "secret_access_key": s3_connection_profile.secret_key,
            }
        )
        create_storage_location = (
            "SELECT pgfs.create_storage_location('images_s3_little', 's3://edb-aidb',"
            f"options => '{options}', "
            f"credentials => '{credentials}'); "
        )
        cur.execute(create_storage_location)

        # Now create a volume from a PGFS storage location for use as a data source in retrievers.
        create_volume = f"SELECT aidb.create_volume('images_bucket_vol', 'images_s3_little', '{s3_connection_profile.recommender_images_path}/', 'Image');"
        cur.execute(create_volume)

        # Now let's define the model we'll use to generate embeddings

        define_model = "SELECT aidb.create_model('recom_images', 'clip_local');"

        cur.execute(define_model)

        create_retriever = (
            "SELECT aidb.create_volume_knowledge_base("
            "name => 'recom_images',"
            "model_name => 'recom_images',"
            "source_volume_name => 'images_bucket_vol',"
            "batch_size => 500);"
        )
        cur.execute(create_retriever)

In [ ]:
def create_product_description_retriever(conn):

    with conn.cursor() as cur:
        # Run retriever for products table
        # The idea is to create a retriever for the products table so the text search can run over it.

        try:
            cur.execute("SAVEPOINT risky_operation")
            cur.execute("SELECT aidb.delete_retriever('recommend_products');")
        except Exception as e:
            cur.execute("ROLLBACK TO SAVEPOINT risky_operation") 

        cur.execute("DROP TABLE IF EXISTS recommend_products_vector CASCADE;")

        #Now let's define the model we'll use to generate embeddings
        embedding_endpoint =  os.environ.get('TEXT_EMBEDDING_ENDPOINT')
        model_name = os.environ.get('TEXT_EMBEDDING_MODEL_NAME')
        text_embedding_service = f"{embedding_endpoint}/v1/embeddings"

        drop_model_sql = "SELECT aidb.delete_model('product_descriptions_embeddings');"
        print(f"Drop model: {drop_model_sql}")
        cur.execute(drop_model_sql)

        # Create a model for product descriptions embeddings
        model_config = json.dumps({
            "model": model_name,
            "url": text_embedding_service
        })
        define_model = (
            "select aidb.create_model("
            "'product_descriptions_embeddings',"
            "'embeddings',"
            f"'{model_config}'::JSONB, "
            "); "
        )
        print(f"Create Model: {define_model}")

        cur.execute(define_model)
        # Create a retriever for product descriptions
        # auto_processing can be set to 'Live', 'Background', or 'Disabled'
        # For more please visit: https://www.enterprisedb.com/docs/edb-postgres-ai/ai-factory/pipeline/knowledge_base/concepts/#consistency-with-source-data
        product_retriever = """SELECT aidb.create_table_knowledge_base(
                    name => 'recommend_products',
                    model_name => 'product_descriptions_embeddings',
                    source_table => 'products',
                    source_key_column => 'product_id',
                    source_data_column => 'productdisplayname',
                    source_data_format => 'Text',
                    auto_processing =>'Live',
                    batch_size => 1000
                    );"""

        print(f"Product Retriever: {product_retriever}")

        cur.execute(product_retriever)

In [ ]:
def create_product_review_model(conn):
    with conn.cursor() as cur:
        # This is the GenAI model that will be used to generate review summary
        try:
            cur.execute("SAVEPOINT risky_operation")

            generative_endpoint =  os.environ.get('GENERATIVE_ENDPOINT')
            model_name = os.environ.get('GENERATIVE_MODEL_NAME')
            generative_service = f"{generative_endpoint}/v1/chat/completions"

            drop_model_sql = "SELECT aidb.delete_model('product_review_model');"
            print(f"Drop model: {drop_model_sql}")
            cur.execute(drop_model_sql)
            genai_config = json.dumps({
            "model": "llama-31-8b-instruct"
            ,"url": "https://llama-31-8b-instruct-samouelian-edb-ai.apps.ai-dev01.kni.syseng.devcluster.openshift.com/v1/chat/completions"
            })
            cur.execute(
                        f"""select aidb.create_model('product_review_model', 'completions',
                        '{genai_config}'::JSONB);"""
                    )
        except Exception as e:
            cur.execute("ROLLBACK TO SAVEPOINT risky_operation") 


In [ ]:
try:
    conn = connect_db()
    create_product_description_retriever(conn)
    create_image_retriever(conn)
    create_product_review_model(conn)
    conn.commit()
except Exception as e:
    conn.rollback()
    raise(e)
finally:
    close_db(conn)